# THE SHAZAM ALGORITHM
This notebook recreates the "Shazam" algorithm, inspired by [this paper](https://www.ee.columbia.edu/~dpwe/papers/Wang03-shazam.pdf).

In [ ]:
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import maximum_filter
from scipy.signal import spectrogram, butter, filtfilt
from collections import defaultdict
import subprocess
import soundfile as sf
import os

## Audio Loading and Spectrogram Generation
Here we mostly set things up, setting up a main class and initialize methods we'll use repeatedly. We set up methods for:
* Loading audio files
* Computing Spectrograms

In [ ]:
class ShazamSimple:
    def __init__(self, sample_rate=22050):
        """
        Shazam-like audio fingerprinting

        Parameters:
        - sample_rate: Audio sample rate (Hz)
        """
        self.sample_rate = sample_rate
        self.database = {}  # Will store {song_id: fingerprints}
        self.song_names = {}  # Will store {song_id: song_name}

    def load_audio(self, filepath):
        """Load audio file and return audio samples"""
        audio, sr = librosa.load(filepath, sr=self.sample_rate, mono=True)
        return audio

    def compute_spectrogram(self, audio):
        """
        Compute spectrogram of audio signal

        Returns:
        - freqs: Frequency bins
        - times: Time bins
        - Sxx: Spectrogram magnitude (in dB)
        """
        # Compute STFT
        n_fft = 2048
        hop_length = 512

        S = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length)
        S_db = librosa.amplitude_to_db(np.abs(S), ref=np.max)

        freqs = librosa.fft_frequencies(sr=self.sample_rate, n_fft=n_fft)
        times = librosa.frames_to_time(
            np.arange(S_db.shape[1]),
            sr=self.sample_rate,
            hop_length=hop_length
        )

        return freqs, times, S_db

# Initialize the system
shazam = ShazamSimple(sample_rate=22050)

## Peak Detection in Constellation Map
This code defines functions to find the peaks between "high points" in the spectogram.
* We can define the **threshold** for these peaks (how loud it needs to be to be considered a peak)
* We can define the **minimum distance** between these peaks to ensure we don't get peaks all clustered together in one loud part of the spectrogram.

In [ ]:
def find_peaks(self, spectrogram, min_distance=10, threshold=-60):
    """
    Find peaks in spectrogram to create constellation map

    Parameters:
    - spectrogram: 2D array (frequency x time)
    - min_distance: Minimum distance between peaks
    - threshold: Minimum dB threshold for a peak

    Returns:
    - peaks: List of (time_idx, freq_idx) tuples
    """
    # Apply maximum filter to find local maxima
    neighborhood_size = (100, 100)  # (frequency bins, time bins)
    local_max = maximum_filter(spectrogram, size=neighborhood_size)

    # Peak is where spectrogram equals local maximum and exceeds threshold
    peaks = (spectrogram == local_max) & (spectrogram > threshold)

    # Get coordinates of peaks
    peak_coords = np.argwhere(peaks)

    # Sort by amplitude (descending) to prioritize strongest peaks
    peak_amplitudes = spectrogram[peaks]
    sorted_indices = np.argsort(peak_amplitudes)[::-1]
    peak_coords = peak_coords[sorted_indices]

    return peak_coords

# Add method to class
ShazamSimple.find_peaks = find_peaks


## Create Constellation Map
This code essentially combines the previous steps into a constellation map. We define functions to define the spectrogram, find the peaks, then plot the resultant constellation map.

In [ ]:
def create_constellation_map(self, audio, max_peaks=200, plot=True):
    """
    Create constellation map from audio

    Parameters:
    - audio: Audio samples
    - max_peaks: Maximum number of peaks to keep
    - plot: Whether to visualize the constellation map

    Returns:
    - constellation: List of (time_idx, freq_idx) peak coordinates
    - freqs: Frequency bins
    - times: Time bins
    - spectrogram: The computed spectrogram
    """
    # Compute spectrogram
    freqs, times, spec_db = self.compute_spectrogram(audio)

    # Find peaks
    peaks = self.find_peaks(spec_db, min_distance=10, threshold=-60)

    # Limit number of peaks
    if len(peaks) > max_peaks:
        peaks = peaks[:max_peaks]

    # Convert to list of tuples (time_idx, freq_idx)
    constellation = [(int(p[1]), int(p[0])) for p in peaks]

    if plot:
        self.plot_constellation(spec_db, constellation, freqs, times)

    return constellation, freqs, times, spec_db

def plot_constellation(self, spectrogram, constellation, freqs, times):
    fig, (ax_spec, ax_const) = plt.subplots(1, 2, figsize=(14, 6))

    # ---- Spectrogram ----
    librosa.display.specshow(
        spectrogram,
        sr=self.sample_rate,
        x_axis='time',
        y_axis='hz',
        cmap='viridis',
        ax=ax_spec
    )
    fig.colorbar(ax_spec.collections[0], ax=ax_spec, format='%+2.0f dB')
    ax_spec.set_title('Spectrogram')
    ax_spec.set_ylim(0, 8000)

    # ---- Constellation ----
    if len(constellation) > 0:
        time_indices, freq_indices = zip(*constellation)
        freqs_used = freqs[list(freq_indices)]

        max_freq = freqs_used.max()
        padding_hz = 300
        y_max = min(max_freq + padding_hz, freqs[-1])

        ax_const.scatter(
            times[list(time_indices)],
            freqs_used,
            c='red',
            s=10,
            alpha=0.6,
            marker='x'
        )
        ax_const.set_ylim(0, y_max)
    else:
        ax_const.set_ylim(0, 2000)

    ax_const.set_title(f'Constellation Map ({len(constellation)} peaks)')
    ax_const.set_xlabel('Time (s)')
    ax_const.set_ylabel('Frequency (Hz)')
    ax_const.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Add methods to class
ShazamSimple.create_constellation_map = create_constellation_map
ShazamSimple.plot_constellation = plot_constellation

## Generate Fingerprints
Creates fingerprints by selecting pairs of points and encoding the two frequencies and their differences (in time) into a hash.

In [ ]:
from matplotlib.patches import Rectangle

def generate_fingerprints(
    self,
    constellation,
    fan_value=5,
    delta_t_min=5,
    delta_t_max=200,
    delta_f_max=90
):
    """
    Generate fingerprints using combinatorial hashing (Shazam-style)

    Each anchor point is paired with several target points within
    bounded time and frequency windows, producing hashes of the form
    (freq1, freq2, delta_t).

    Parameters:
    - constellation: List of (time_idx, freq_idx) tuples
    - fan_value: Number of target points to pair with each anchor
    - delta_t_min: Minimum time difference (frames)
    - delta_t_max: Maximum time difference (frames)
    - delta_f_max: Maximum frequency difference (bins)

    Returns:
    - fingerprints: List of (hash, anchor_time) tuples
    """

    fingerprints = []

    # Sort constellation points by time index
    constellation_sorted = sorted(constellation, key=lambda x: x[0])

    for i, (t1, f1) in enumerate(constellation_sorted):

        # Look ahead to the next fan_value peaks in time order
        for j in range(i + 1, min(i + fan_value + 1, len(constellation_sorted))):
            t2, f2 = constellation_sorted[j]

            delta_t = t2 - t1
            delta_f = abs(f2 - f1)

            # Bounded time–frequency target zone
            if (
                delta_t_min <= delta_t <= delta_t_max
                and delta_f <= delta_f_max
            ):
                hash_value = (f1, f2, delta_t)
                fingerprints.append((hash_value, t1))

    return fingerprints


# Add method to class
ShazamSimple.generate_fingerprints = generate_fingerprints


def plot_single_anchor_fan(
    self,
    constellation,
    fingerprints,
    freqs,
    times,
    anchor_index,
    delta_t_min=5,
    delta_t_max=200,
    delta_f_max=50
):
    """
    Visualize the fingerprint fan for a single anchor point,
    overlaid on the constellation map, including the
    time–frequency target window.
    """

    fig, ax = plt.subplots(figsize=(10, 6))

    # ---- Plot constellation map ----
    time_idx, freq_idx = zip(*constellation)
    ax.scatter(
        times[list(time_idx)],
        freqs[list(freq_idx)],
        c='red',
        s=20,
        marker='x',
        alpha=0.6,
        label='Constellation peaks'
    )

    # ---- Anchor point ----
    t_anchor, f_anchor = constellation[anchor_index]
    ax.scatter(
        times[t_anchor],
        freqs[f_anchor],
        s=120,
        c='gold',
        edgecolors='black',
        zorder=5,
        label='Anchor'
    )

    # ---- Target window (time–frequency box) ----
    t_start = times[t_anchor + delta_t_min]
    t_end = times[min(t_anchor + delta_t_max, len(times) - 1)]

    f_center = freqs[f_anchor]
    f_min = max(f_center - delta_f_max * (freqs[1] - freqs[0]), 0)
    f_max = f_center + delta_f_max * (freqs[1] - freqs[0])

    window = Rectangle(
        (t_start, f_min),
        t_end - t_start,
        f_max - f_min,
        linewidth=1.5,
        edgecolor='tab:green',
        facecolor='none',
        linestyle='--',
        label='Target window'
    )

    ax.add_patch(window)

    # ---- Target points and fingerprint connections ----
    for (f1, f2, delta_t), t1 in fingerprints:
        if t1 == t_anchor and f1 == f_anchor:
            t2 = t1 + delta_t

            ax.plot(
                [times[t1], times[t2]],
                [freqs[f1], freqs[f2]],
                color='tab:blue',
                linewidth=1.5,
                alpha=0.8
            )

            ax.scatter(
                times[t2],
                freqs[f2],
                s=50,
                c='tab:blue',
                marker='o',
                zorder=4
            )

    ax.set_title("Single-Anchor Fingerprint with Target Zone")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.set_ylim(0, 6000)
    ax.legend()
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.show()


# Add method to class
ShazamSimple.plot_single_anchor_fan = plot_single_anchor_fan

## Database generation
This code sets up the function to process songs and get them into the database (backend).

In [ ]:
def add_song_to_database(self, filepath, song_id, max_peaks=200):
    """
    Process a song and add its fingerprints to the database

    Parameters:
    - filepath: Path to audio file
    - song_id: Unique identifier for the song
    - max_peaks: Maximum peaks in constellation map
    """
    print(f"Processing: {song_id}")

    # Load audio
    audio = self.load_audio(filepath)

    # Create constellation map
    constellation, freqs, times, spec = self.create_constellation_map(
        audio,
        max_peaks=max_peaks,
        plot=True
    )

    # Generate fingerprints
    fingerprints = self.generate_fingerprints(constellation, fan_value=5)

    # Store in database
    self.database[song_id] = fingerprints
    self.song_names[song_id] = os.path.basename(filepath)

    print(f"  → Generated {len(fingerprints)} fingerprints")
    print(f"  → Duration: {len(audio)/self.sample_rate:.2f} seconds")
    print()

# Add method to class
ShazamSimple.add_song_to_database = add_song_to_database
print("Database building functions added")

## Audio Matching
This code sets up the function to actually compare a given audio clip against the database - it handles generating the new constellation map and fingerprints, and then checks against the database.

In [ ]:
def match_audio(self, audio_clip, plot_results=True):
    """
    Match an audio clip against the database

    Parameters:
    - audio_clip: Audio samples to identify
    - plot_results: Whether to plot matching visualization

    Returns:
    - best_match: (song_id, confidence_score)
    """
    # Generate constellation and fingerprints for query
    constellation, _, _, _ = self.create_constellation_map(
        audio_clip,
        max_peaks=200,
        plot=plot_results
    )
    query_fingerprints = self.generate_fingerprints(constellation, fan_value=5)

    print(f"Query has {len(query_fingerprints)} fingerprints")

    # Create hash lookup for query
    query_hashes = {fp[0]: fp[1] for fp in query_fingerprints}

    # Match against each song in database
    matches = defaultdict(list)  # {song_id: [(query_time, db_time), ...]}

    for song_id, db_fingerprints in self.database.items():
        for db_hash, db_time in db_fingerprints:
            if db_hash in query_hashes:
                query_time = query_hashes[db_hash]
                matches[song_id].append((query_time, db_time))

    # Score each song by finding time-aligned matches
    scores = {}
    for song_id, time_pairs in matches.items():
        if len(time_pairs) < 5:  # Minimum threshold
            continue

        # Calculate time differences (should cluster if it's a match)
        time_diffs = [db_time - query_time for query_time, db_time in time_pairs]

        # Find most common time difference (mode)
        diff_histogram = defaultdict(int)
        for diff in time_diffs:
            diff_histogram[diff] += 1

        # Best alignment is the most frequent time difference
        best_alignment = max(diff_histogram, key=diff_histogram.get)
        score = diff_histogram[best_alignment]

        scores[song_id] = {
            'score': score,
            'alignment': best_alignment,
            'time_pairs': time_pairs
        }

    if not scores:
        print("No matches found!")
        return None, 0

    # Get best match
    best_match = max(scores, key=lambda x: scores[x]['score'])
    best_score = scores[best_match]['score']

    print(f"\n🎵 Best Match: {self.song_names[best_match]}")
    print(f"   Confidence: {best_score} aligned fingerprints")

    if plot_results:
        self.plot_match_results(best_match, scores[best_match])

    return best_match, best_score

def plot_match_results(self, song_id, match_info):
    """Visualize the time alignment of matches"""
    time_pairs = match_info['time_pairs']
    alignment = match_info['alignment']

    if not time_pairs:
        return

    query_times, db_times = zip(*time_pairs)

    plt.figure(figsize=(10, 6))
    plt.scatter(query_times, db_times, alpha=0.5, s=20)
    plt.xlabel('Query Time (time bins)')
    plt.ylabel('Database Time (time bins)')
    plt.title(f'Time Alignment Scatter Plot - {len(time_pairs)} matches')
    plt.grid(alpha=0.3)

    # Draw alignment line
    x_range = [min(query_times), max(query_times)]
    plt.plot(x_range, [x + alignment for x in x_range], 'r--', alpha=0.7, label=f'Alignment offset: {alignment}')
    plt.legend()

    plt.tight_layout()
    plt.show()

# Add methods to class
ShazamSimple.match_audio = match_audio
ShazamSimple.plot_match_results = plot_match_results

## Visualization (Optional)
Quick Visualization for a given song, seeing the spectrogram, constellation map, and fingerprint.


In [ ]:
audio = shazam.load_audio("CT.mp3")

# Step 1: constellation
constellation, freqs, times, spec = shazam.create_constellation_map(
    audio,
    max_peaks=200,
    plot=True
)

fingerprints = shazam.generate_fingerprints(
    constellation,
    fan_value=20,
    delta_t_min=100,
    delta_t_max=900,
    delta_f_max=60
)

# Pick any anchor index you like
shazam.plot_single_anchor_fan(
    constellation,
    fingerprints,
    freqs,
    times,
    anchor_index=25,
    delta_t_min=100,
    delta_t_max=900,
    delta_f_max=60
)


## Database population
Here we actually populate our database with songs, using a lot of the previous functions.

Note the spectrogram and constellation map of each song!

In [ ]:
# Build the database with multiple songs
# You'll need to upload your audio files to the notebook first (if using colab)

# file paths
songs = [
    "CT.mp3"
]

# Add songs to database
print("=" * 60)
print("BUILDING DATABASE")
print("=" * 60)

for i, song_path in enumerate(songs):
    if os.path.exists(song_path):
        shazam.add_song_to_database(song_path, song_id=f"song_{i}", max_peaks=200)
    else:
        print(f"Warning: {song_path} not found - skipping")

print(f"\nDatabase contains {len(shazam.database)} songs")

## Song Identification
In the code below, we take a 60 second clip from the middle of the selected audio clip.

We generate a constellation map for the new clip and search against the existing constellation maps in the database.

The graph at the very end plots matches over time. We want to see a straight, diagonal line - this shows us that fingerprint points are matching over time with a consistent offset between the test clip and the database item.

In [ ]:
# Test with a clip from one of the songs
print("\n" + "=" * 60)
print("TESTING RECOGNITION")
print("=" * 60)

# Load a test clip (could be from middle of a song, with noise, etc.)
test_clip_path = "CT.mp3"  # Replace with your test file

if os.path.exists(test_clip_path):
    test_audio = shazam.load_audio(test_clip_path)

    # extract just a portion
    clip_duration = 45  # seconds
    clip_samples = clip_duration * shazam.sample_rate
    if len(test_audio) > clip_samples:
        start_idx = len(test_audio) // 2
        test_audio = test_audio[start_idx:start_idx + clip_samples]

    # Match against database
    matched_song, confidence = shazam.match_audio(test_audio, plot_results=True)
else:
    print(f"Test file {test_clip_path} not found")

## Song Identification (plus noise)
A real test of algorithmic robustness - does it still work when we introduce noise to the audio signal?

In [ ]:
# Test with a clip from one of the songs
print("\n" + "=" * 60)
print("TESTING RECOGNITION")
print("=" * 60)

# Load a test clip (could be from middle of a song, with noise, etc.)
test_clip_path = "CT.mp3"  # Replace with your test file

if os.path.exists(test_clip_path):
    test_audio = shazam.load_audio(test_clip_path)

    # extract just a portion
    clip_duration = 45  # seconds
    clip_samples = clip_duration * shazam.sample_rate
    if len(test_audio) > clip_samples:
        start_idx = len(test_audio) // 2
        test_audio = test_audio[start_idx:start_idx + clip_samples]


    noise_level = 0.025  # adjust this (0.005–0.02 works well)

    # Generate white noise
    white_noise = np.random.randn(len(test_audio))

    # Band-pass filter settings (focus on musically relevant frequencies)
    low_hz = 50
    high_hz = 4000
    nyquist = shazam.sample_rate / 2

    b, a = butter(
        N=4,
        Wn=[low_hz / nyquist, high_hz / nyquist],
        btype='band'
    )

    # Filter the noise to make it "colored"
    colored_noise = filtfilt(b, a, white_noise)

    # Normalize and scale
    colored_noise /= np.std(colored_noise)
    colored_noise *= noise_level

    test_audio = test_audio + colored_noise

    # Prevent clipping if audio is normalized to [-1, 1]
    test_audio = np.clip(test_audio, -1.0, 1.0)

    # Match against database
    matched_song, confidence = shazam.match_audio(
        test_audio,
        plot_results=True
    )
else:
    print(f"Test file {test_clip_path} not found")
